In [1]:
import argparse
import os,cv2
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import argparse
import os
import random,numpy
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

Random Seed:  999


In [4]:
train_info=pandas.read_csv("/kaggle/input/cidaut-ai-fake-scene-classification-2024/train.csv")

if os.path.exists(f"/kaggle/working/train/")==False:
    os.mkdir(f"/kaggle/working/train/")
    os.mkdir(f"/kaggle/working/train/real/")
    os.mkdir(f"/kaggle/working/train/editada/")
    os.mkdir(f"/kaggle/working/test/")
    os.mkdir(f"/kaggle/working/test/real/")
    os.mkdir(f"/kaggle/working/test/editada/")

for i in zip(train_info['image'],train_info['label'],range(len(train_info['image']))):
    
    if i[2]<=69:
        if i[1]=="real":
            cv2.imwrite(f"/kaggle/working/test/real/{i[0]}",cv2.resize(cv2.imread(f"/kaggle/input/cidaut-ai-fake-scene-classification-2024/Train/{i[0]}"),(256,256)))
        elif i[1]=="editada":
            cv2.imwrite(f"/kaggle/working/test/editada/{i[0]}",cv2.resize(cv2.imread(f"/kaggle/input/cidaut-ai-fake-scene-classification-2024/Train/{i[0]}"),(256,256)))

    
    if i[1]=="real":
        cv2.imwrite(f"/kaggle/working/train/real/{i[0]}",cv2.resize(cv2.imread(f"/kaggle/input/cidaut-ai-fake-scene-classification-2024/Train/{i[0]}"),(256,256)))
    elif i[1]=="editada":
        cv2.imwrite(f"/kaggle/working/train/editada/{i[0]}",cv2.resize(cv2.imread(f"/kaggle/input/cidaut-ai-fake-scene-classification-2024/Train/{i[0]}"),(256,256)))
        

In [5]:
workers = 2
batch_size = 10
nz = 100
num_epochs = 5
lr = 0.0001
beta1 = 0.5
ngpu=1
ngf,nc = 3,3
ndf = 64

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

trainset = torchvision.datasets.ImageFolder(root=f'/kaggle/working/train',transform=transform)
dataloader = torch.utils.data.DataLoader(trainset,batch_size=batch_size,shuffle=True,num_workers=2)

testset = torchvision.datasets.ImageFolder(root=f'/kaggle/working/test',transform=transform)
testloader = torch.utils.data.DataLoader(testset,batch_size=batch_size,shuffle=True,num_workers=2)

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

In [6]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

class FSC(nn.Module):
    def __init__(self):
        super().__init__()
        self.FSC_rafire=nn.Sequential(
            nn.Conv2d(3, 16, 3),  # Increased channels
            nn.BatchNorm2d(16),
            nn.LeakyReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(16, 64, 3),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 126, 3),
            nn.BatchNorm2d(126),
            nn.LeakyReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(126, 256, 3),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(256, 512, 3),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(512, 1024, 3),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Flatten(),

            nn.Linear(4096, 1024),
            #nn.BatchNorm1d(128),
            nn.Dropout(0.1),
            nn.LeakyReLU(),

            nn.Linear(1024, 512),
            #nn.BatchNorm1d(64),
            nn.Dropout(0.1),
            nn.LeakyReLU(),

            nn.Linear(512, 2)  
        )

    def forward(self,x):
        rsc_rafire = self.FSC_rafire(x)
        return  rsc_rafire,torch.argmax(rsc_rafire, dim=1)

In [7]:
FSC_net = FSC().to(device)
if (device.type == 'cuda') and (ngpu > 1):
    FSC_net = nn.DataParallel(FSC_net, list(range(ngpu)))
FSC_net.apply(weights_init)

FSC(
  (FSC_rafire): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.01)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(16, 64, kernel_size=(3, 3), stride=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): LeakyReLU(negative_slope=0.01)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 126, kernel_size=(3, 3), stride=(1, 1))
    (9): BatchNorm2d(126, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): LeakyReLU(negative_slope=0.01)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(126, 256, kernel_size=(3, 3), stride=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=T

In [8]:
criterion = nn.CrossEntropyLoss()

# CrossEntropyLoss
# MSELoss
# L1Loss
# BCELoss
# BCEWithLogitsLoss

optimizer = optim.AdamW(FSC_net.parameters(), lr=lr, betas=(beta1, 0.999))
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.86)

In [9]:
#netD.load_state_dict(torch.load("/kaggle/input/rafire_4/pytorch/default/1/rafire.pth"))
print(dataloader.dataset.classes)
z_w_=[]
high_rf,i_w,z_k,correct=0,0,0,0

lab={
    'editada' : 0,
    'real' : 1
}

while(i_w<4000):
    z_,z,z_w=0,0,{}
    for i, data in enumerate(dataloader, 0):
        real=data[0].to(device)
        label=data[1].to(device) #
        #print(data[0].shape)
        raw_label=torch.from_numpy(numpy.array([[0,1] if i==1 else [1,0] for i in data[1] ])).float().to(device)
        raw_output,output = FSC_net(real)
        raw_output,output = raw_output.float() ,torch.tensor([lab[dataloader.dataset.classes[i]] for i in output.view(-1)])
        #print(raw_label,raw_output)
        err_real = criterion(raw_output,raw_label) #err_real.requires_grad = True
        optimizer.zero_grad()
        err_real.backward()
        optimizer.step()
        
    for i, data in enumerate(testloader, 0):
        real=data[0].to(device)
        label=data[1].to(device)
        raw_output,output = FSC_net(real)
        output=torch.tensor([lab[dataloader.dataset.classes[i]] for i in output.view(-1)])
        z_+=len(output)
        correct+=(output==label).float().sum()
        
    acc=100*correct/len(trainset)
    z_w_.append(acc)
    
    print(output,label)
    print(f"EPOCH: {z_k} LOSS_FSC: {err_real.item()} ACCURACY: {acc}")
    #print(z_)	
        
    if len(z_w_)>=3:
        if len([True for i in range(1,4) if z_w_[len(z_w_)-i]<=z_w_[len(z_w_)-3] and z_w_[len(z_w_)-i]>=z_w_[len(z_w_)-4]])==3:
            z_w_=[]
            print(optimizer.param_groups[0]["lr"])
            scheduler.step()
            print(optimizer.param_groups[0]["lr"])
            
    if acc>high_rf:
        high_rf=acc
        torch.save(FSC_net.state_dict(),f'/kaggle/working/{acc}.pth')
    if z_k==100:
        break
    z_k+=1
    i_w+=1
    correct=0

['editada', 'real']
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1]) tensor([1, 0, 1, 1, 1, 1, 1, 0, 1, 1])
EPOCH: 0 LOSS_FSC: 0.8643363118171692 ACCURACY: 5.277777671813965
tensor([1, 1, 1, 1, 1, 0, 1, 0, 0, 1]) tensor([1, 0, 0, 0, 0, 0, 1, 0, 0, 1])
EPOCH: 1 LOSS_FSC: 0.6901310682296753 ACCURACY: 6.80555534362793
tensor([1, 1, 1, 0, 1, 0, 0, 0, 1, 1]) tensor([1, 0, 0, 1, 1, 1, 0, 0, 1, 0])
EPOCH: 2 LOSS_FSC: 0.5998005867004395 ACCURACY: 5.833333492279053
tensor([0, 0, 1, 0, 1, 1, 1, 1, 1, 1]) tensor([0, 0, 1, 0, 1, 1, 0, 1, 0, 1])
EPOCH: 3 LOSS_FSC: 0.588723361492157 ACCURACY: 6.666666507720947
0.0001
8.6e-05
tensor([0, 1, 0, 1, 1, 0, 0, 0, 0, 1]) tensor([1, 1, 0, 1, 1, 0, 1, 0, 1, 1])
EPOCH: 4 LOSS_FSC: 0.5445915460586548 ACCURACY: 7.361111164093018
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1]) tensor([0, 1, 0, 1, 1, 1, 0, 1, 1, 1])
EPOCH: 5 LOSS_FSC: 0.6733163595199585 ACCURACY: 6.527777671813965
tensor([1, 1, 0, 0, 1, 0, 1, 0, 0, 1]) tensor([1, 0, 0, 1, 1, 1, 0, 0, 0, 0])
EPOCH: 6 LOSS_FSC: 0.4373